In [0]:
import data.utilities.common as Commonlib

import pyspark.sql.functions as F
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.types import TimestampType
from pyspark.sql.window import Window

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
rtl_acct_grp = MTU.get_fully_qualified_table_name("oracle", "mss", "rtl_acct_grp")

In [0]:
rtl_acct_grp_df = spark.read.table(rtl_acct_grp).select(
    F.col("rtl_acct_grp_id").alias("chain_acct_nbr"),
    F.expr(
        "case when rtl_acct_grp_typ_cd = 'NCA' then 'NCHAIN' when rtl_acct_grp_typ_cd = 'ASSN' then 'RTLASCN' else rtl_acct_grp_typ_cd end"
    ).alias("chain_typ_cd"),
    F.col("rtl_acct_grp_nm").alias("chain_nm"),
    F.col("addr_txt").alias("chain_addr_txt"),
    F.col("city_nm").alias("chain_city_nm"),
    F.col("st_abbr").alias("chain_st_cd"),
    F.col("zip").alias("chain_zip_cd"),
    F.col("phone_area_cd").alias("chain_area_cd"),
    F.col("phone_nbr").alias("chain_phone_nbr"),
    F.col("beg_eff_dt").alias("eff_start_tsp"),
    F.col("end_eff_dt").alias("eff_end_tsp"),
    F.lit(medallion.start_datetime).cast(TimestampType()).alias("__created_tsp"),
)


In [0]:
medallion.df = rtl_acct_grp_df

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="upsert", layer="gold")
except Exception as e:
    medallion.logger.error(f"Error writing chain table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()

except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )